# Bundle Migration: Export & Transform (CSV-Driven)

## Overview

Export and transform dashboards based on inventory CSV from Bundle_00.

**Prerequisites:**
- ✅ Bundle_00 completed (inventory.csv generated)
- ✅ Config file updated with target workspace details

**What this notebook does:**
1. Loads dashboard inventory from CSV (from Bundle_00)
2. Optionally filters dashboards based on user-provided filter CSV
3. Exports dashboard JSONs and permissions
4. Applies catalog/schema transformations
5. Saves transformed files for bundle generation

**Next step:** Run `Bundle_02_Generate_and_Deploy.ipynb`

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
import sys
import os
from pathlib import Path

# Dynamically locate helpers directory
print("=== HELPERS PATH RESOLUTION DEBUG ===")

try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(f"Notebook path: {notebook_path}")
    
    # Get parent directory of Bundle folder
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    print(f"Notebook dir: {notebook_dir}")
    print(f"Bundle parent: {bundle_parent}")
    
    # Verify helpers exists in bundle_parent
    helpers_path = os.path.join(bundle_parent, 'helpers')
    print(f"\nChecking helpers path: {helpers_path}")
    
    if os.path.exists(helpers_path):
        print(f"  Helpers directory EXISTS")
        init_file = os.path.join(helpers_path, '__init__.py')
        if os.path.exists(init_file):
            print(f"  __init__.py FOUND")
            print(f"  Contents: {os.listdir(helpers_path)[:10]}")
        else:
            print(f"  WARNING: No __init__.py found")
    else:
        print(f"  WARNING: Helpers directory does not exist")
        print(f"  Bundle parent contents: {os.listdir(bundle_parent)}")
    
    # Add BUNDLE PARENT to sys.path (not the helpers dir itself)
    sys_path_entry = bundle_parent
    
except Exception as e:
    print(f"Error in workspace context: {e}")
    import traceback
    traceback.print_exc()
    # Fallback for local execution
    sys_path_entry = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Using local fallback: {sys_path_entry}")

sys_path_entry = os.path.normpath(sys_path_entry)

# Add bundle parent to sys.path (NOT the helpers directory)
print(f"\nAdding to sys.path: {sys_path_entry}")
if sys_path_entry not in sys.path:
    sys.path.insert(0, sys_path_entry)
    print(f"  Added to sys.path")
else:
    print(f"  Already in sys.path")

print(f"\nsys.path first 3 entries: {sys.path[:3]}")
print("=== END DEBUG ===\n")

from helpers import (
    load_config,
    get_path,
    write_volume_file,
    ensure_directory_exists,
    create_workspace_client
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration
config = load_config('../config/config.yaml')

print("\n✅ Configuration loaded\n")
print(f"   Source: {config['source']['workspace_url']}")
print(f"   Volume base: {config['paths']['volume_base']}")

# Ensure directories exist
export_path = get_path('exported')
transformed_path = get_path('transformed')
inventory_path = get_path('inventory')

print("\n📁 Ensuring directories exist...")
ensure_directory_exists(export_path)
ensure_directory_exists(transformed_path)
print(f"   ✅ Export: {export_path}")
print(f"   ✅ Transformed: {transformed_path}")

## Cell 3: Load Inventory from CSV

Load dashboard inventory generated by Bundle_00

In [ ]:
print("\n" + "="*80)
print("STEP 1: LOAD INVENTORY FROM CSV")
print("="*80)

# Load dashboard inventory from volume
inventory_csv = f"{inventory_path}/inventory.csv"

print(f"\n📋 Loading inventory from: {inventory_csv}")

try:
    dashboard_inventory = read_csv_from_volume(inventory_csv)
    print(f"✅ Loaded {len(dashboard_inventory)} dashboards from inventory")
    
    # Display inventory summary
    df = pd.DataFrame(dashboard_inventory)
    print(f"\n📊 Inventory Summary:")
    print(f"   Total dashboards: {len(df)}")
    
    if 'complexity' in df.columns:
        print(f"\n   Complexity breakdown:")
        for complexity, count in df['complexity'].value_counts().items():
            print(f"     {complexity}: {count}")
    
    if 'activity_level' in df.columns:
        print(f"\n   Activity breakdown:")
        for activity, count in df['activity_level'].value_counts().items():
            print(f"     {activity}: {count}")
    
    # Show sample
    print(f"\n📋 Sample dashboards:")
    cols_to_show = ['dashboard_id', 'entity_type', 'table_count', 'complexity', 'activity_level']
    cols_available = [c for c in cols_to_show if c in df.columns]
    display(df[cols_available].head(10))
    
except Exception as e:
    print(f"\n❌ Failed to load inventory CSV: {e}")
    print(f"\n⚠️  Make sure Bundle_00_Inventory_Generation has been run first")
    print(f"   Expected file: {inventory_csv}")
    raise

## Cell 4: Optional - Apply Dashboard Filter

Filter to specific dashboards if filter CSV is provided

In [ ]:
print("\n" + "="*80)
print("STEP 2: APPLY DASHBOARD FILTER (OPTIONAL)")
print("="*80)

# Optional: Load filter CSV if provided
filter_csv_path = f"{inventory_path}/dashboards_to_migrate.csv"

print(f"\n🔍 Checking for filter CSV: {filter_csv_path}")

try:
    filter_list = read_csv_from_volume(filter_csv_path)
    filter_ids = {d['dashboard_id'] for d in filter_list}
    
    original_count = len(dashboard_inventory)
    dashboard_inventory = [d for d in dashboard_inventory if d['dashboard_id'] in filter_ids]
    
    print(f"✅ Filter CSV found")
    print(f"   Original: {original_count} dashboards")
    print(f"   Filtered: {len(dashboard_inventory)} dashboards")
    print(f"   Excluded: {original_count - len(dashboard_inventory)} dashboards")
    
except Exception as e:
    print(f"ℹ️  No filter CSV found - processing all dashboards")
    print(f"   (File: {filter_csv_path})")
    print(f"\n💡 To filter dashboards:")
    print(f"   1. Create CSV with column 'dashboard_id'")
    print(f"   2. Add dashboard IDs you want to migrate")
    print(f"   3. Save to: {filter_csv_path}")
    print(f"   4. Re-run this notebook")

print(f"\n📊 Final selection: {len(dashboard_inventory)} dashboards to process")

## Cell 5: Connect to Source Workspace

In [ ]:
print("\n" + "="*80)
print("STEP 3: CONNECT TO SOURCE WORKSPACE")
print("="*80)

print("\n🔗 Connecting to source workspace...")
source_client = create_workspace_client('source')
print(f"   ✅ Connected to {config['source']['workspace_url']}")

## Cell 6: Export Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("STEP 4: EXPORTING DASHBOARDS & PERMISSIONS")
print("="*80)

if not dashboard_inventory:
    print("\n⚠️  No dashboards to export")
else:
    export_results = []
    
    for i, dash in enumerate(dashboard_inventory, 1):
        dashboard_id = dash['dashboard_id']
        
        # Try to get name from inventory, fallback to ID
        dashboard_name = dash.get('dashboard_name') or dash.get('name') or f"Dashboard_{dashboard_id}"
        
        print(f"\n[{i}/{len(dashboard_inventory)}] {dashboard_name}")
        print(f"   ID: {dashboard_id}")
        
        # Show metadata from inventory
        if 'table_count' in dash:
            print(f"   Tables: {dash['table_count']}, Complexity: {dash.get('complexity', 'Unknown')}")
        
        try:
            # Export dashboard JSON
            json_content, display_name, clean_name = export_dashboard(source_client, dashboard_id)
            
            # Save JSON
            json_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
            write_volume_file(json_file, json_content)
            print(f"   ✅ Exported JSON")
            
            # Get and save permissions
            permissions = get_dashboard_permissions(source_client, dashboard_id)
            permissions['display_name'] = display_name
            
            perm_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
            write_volume_file(perm_file, json.dumps(permissions, indent=2))
            
            acl_count = len(permissions.get('access_control_list', []))
            print(f"   🔐 Permissions: {acl_count} ACL(s)")
            
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': display_name,
                'status': 'success',
                'json_file': json_file
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': dashboard_name,
                'status': 'failed',
                'error': str(e)
            })
    
    successful = len([r for r in export_results if r['status'] == 'success'])
    failed = len([r for r in export_results if r['status'] == 'failed'])
    
    print(f"\n" + "="*80)
    print(f"✅ Successfully exported: {successful}/{len(dashboard_inventory)}")
    if failed > 0:
        print(f"❌ Failed: {failed}")
        print(f"\n⚠️  Failed dashboards:")
        for r in export_results:
            if r['status'] == 'failed':
                print(f"   - {r['name']} ({r['dashboard_id']}): {r.get('error', 'Unknown error')}")

## Cell 7: Transform Dashboards (Catalog/Schema Remapping)

In [ ]:
print("\n" + "="*80)
print("STEP 5: TRANSFORMING DASHBOARDS")
print("="*80)

if not config['transformation']['enabled']:
    print("\n⚠️  Transformation disabled in config")
elif not export_results or successful == 0:
    print("\n⚠️  No dashboards to transform")
else:
    # Load mapping CSV
    mapping_csv_path = get_path('volume_base') + "/" + config['transformation']['mapping_csv']
    print(f"\n📋 Loading mappings: {mapping_csv_path}")
    
    try:
        mappings = load_mapping_csv(mapping_csv_path)
        print(f"   ✅ Loaded {len(mappings)} mapping rule(s)\n")
        
    except Exception as e:
        print(f"   ❌ Failed to load CSV: {e}")
        raise
    
    # Transform each dashboard
    transform_results = []
    successful_exports = [r for r in export_results if r['status'] == 'success']
    
    for i, result in enumerate(successful_exports, 1):
        name = result['name']
        json_file = result['json_file']
        
        print(f"[{i}/{len(successful_exports)}] Transforming: {name}")
        
        try:
            # Read exported JSON
            from helpers.volume_utils import read_volume_file
            json_content = read_volume_file(json_file)
            
            # Apply transformations
            transformed = transform_dashboard_json(json_content, mappings)
            
            # Save transformed version
            filename = Path(json_file).name
            transformed_file = f"{transformed_path}/{filename}"
            write_volume_file(transformed_file, transformed)
            
            print(f"   ✅ Transformed")
            
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'success'
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'failed'
            })
    
    successful_transforms = len([r for r in transform_results if r['status'] == 'success'])
    
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"\n✅ Exported: {successful}/{len(dashboard_inventory)}")
    print(f"✅ Transformed: {successful_transforms}/{successful}")
    print(f"\n📁 Files ready at: {transformed_path}")
    print(f"\n▶️  Next: Run Bundle_02_Generate_and_Deploy.ipynb")